# Ticker/Month Loader Batch Inspection

Use this notebook on the workstation to load one or more batches from the ticker/month rolling cache and inspect identities, tensor shapes, masks, labels, event streams, text tokens, XBRL, daily bars, and loader timing. The defaults are intentionally small enough for interactive inspection.

In [ ]:
from __future__ import annotations

import datetime as dt
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

repo_candidates = [
    Path.cwd(),
    Path("D:/TradingML/codes/quant_research_workbench_pipelines"),
    Path("D:/TradingCodes/quant-research-workbench"),
]
for candidate in repo_candidates:
    if (candidate / "research" / "mlops" / "rolling_loader").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate quant-research-workbench repo root. Set REPO_ROOT manually.")

from research.mlops.rolling_loader.ticker_month_cache import DEFAULT_TICKER_MONTH_CACHE_ROOT
from research.mlops.rolling_loader.ticker_month_dataset import AsyncTickerMonthBatchLoader, TickerMonthLoaderConfig

REPO_ROOT

## Configure The Loader

The notebook auto-selects `train_201902_201912_ticker_month` if it exists, then falls back to the current `train_201902_201907_ticker_month` cache. Change `CACHE_ID`, `MONTHS`, `BATCH_SIZE`, or `TICKERS` here for focused checks.

In [ ]:
CACHE_ROOT_BASE = Path("D:/market-data/prepared/rolling_ticker_month_cache")
CACHE_ID_CANDIDATES = [
    "train_201902_201912_ticker_month",
    "train_201902_201907_ticker_month",
]
def cache_has_packages(cache_id: str) -> bool:
    root = CACHE_ROOT_BASE / cache_id
    manifest_path = root / "manifest.json"
    if not manifest_path.exists():
        return False
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception:
        return False
    if manifest.get("packages"):
        return True
    return any(root.glob("*/month=*/ticker_hash=*/ticker=*/manifest.json"))

CACHE_ID = next((cache_id for cache_id in CACHE_ID_CANDIDATES if cache_has_packages(cache_id)), CACHE_ID_CANDIDATES[0])
CACHE_ROOT = CACHE_ROOT_BASE / CACHE_ID

MONTHS = ("2019-02",)
TICKERS: tuple[str, ...] = ()
BATCH_SIZE = 1024
SEED = 17
DATASET_ID = "inspection_201902_v1"

config = TickerMonthLoaderConfig(
    cache_root=CACHE_ROOT,
    split="train",
    months=MONTHS,
    tickers=TICKERS,
    batch_size=BATCH_SIZE,
    seed=SEED,
    dataset_id=DATASET_ID,
    data_groups=(
        "events",
        "intraday_labels",
        "daily_bars",
        "global_daily_bars",
        "ticker_news_tokens",
        "market_news_tokens",
        "sec_filing_tokens",
        "xbrl",
    ),
    event_output_mode="raw_stream",
    suppress_event_columns=("ticker_id", "ordinal", "timestamp_us"),
    event_stream_length=1024,
    event_stream_chunk_size=128,
    loaded_parts_per_group=4,
    read_workers=2,
    materialize_workers=4,
    materialize_chunk_size=256,
    max_batches=1,
    ticker_news_max_items=8,
    market_news_max_items=16,
    sec_filing_max_items=4,
    xbrl_max_items=4096,
    ticker_news_token_chunks=2,
    market_news_token_chunks=2,
    sec_filing_token_chunks=8,
    text_max_tokens=1024,
    ticker_daily_bar_offsets=(1, 2, 3, 7, 14, 28, 40, 200),
    global_daily_bar_offsets=(1, 2, 7),
)

print(json.dumps({
    "cache_root": str(config.cache_root),
    "exists": config.cache_root.exists(),
    "months": config.months,
    "tickers": config.tickers,
    "batch_size": config.batch_size,
    "data_groups": config.data_groups,
}, indent=2))
if not config.cache_root.exists():
    raise FileNotFoundError(config.cache_root)

In [ ]:
loader = AsyncTickerMonthBatchLoader(config)
print("Loader summary:")
print(json.dumps(loader.summary(), indent=2, sort_keys=True))

batch = next(loader.iter_batches())
print("Loaded batch samples:", batch.sample_count)
print("Batch profile:")
print(json.dumps(batch.profile, indent=2, sort_keys=True))

## Shape Summary

In [ ]:
def array_summary(name: str, value: np.ndarray) -> dict[str, object]:
    arr = np.asarray(value)
    out: dict[str, object] = {"name": name, "shape": tuple(arr.shape), "dtype": str(arr.dtype)}
    if arr.size and np.issubdtype(arr.dtype, np.number):
        finite = arr[np.isfinite(arr)] if np.issubdtype(arr.dtype, np.floating) else arr.reshape(-1)
        if finite.size:
            out.update({"min": float(np.min(finite)), "max": float(np.max(finite)), "mean": float(np.mean(finite))})
        out["zero_fraction"] = float(np.mean(arr == 0))
    elif arr.size and arr.dtype == np.bool_:
        out["true_fraction"] = float(np.mean(arr))
    return out

rows = [
    array_summary("ticker", batch.ticker),
    array_summary("origin_ordinal", batch.origin_ordinal),
    array_summary("origin_timestamp_us", batch.origin_timestamp_us),
    array_summary("source_part_key", batch.source_part_key),
    array_summary("raw_event_stream", batch.raw_event_stream),
    array_summary("raw_event_mask", batch.raw_event_mask),
    array_summary("future_intraday_bars", batch.future_intraday_bars),
    array_summary("future_intraday_bar_mask", batch.future_intraday_bar_mask),
]
for name, values in batch.intraday_labels.items():
    rows.append(array_summary(f"intraday_labels.{name}", values))
for name, payload in batch.text_inputs.items():
    for field, values in payload.items():
        rows.append(array_summary(f"text_inputs.{name}.{field}", values))
for field, values in batch.xbrl_inputs.items():
    rows.append(array_summary(f"xbrl_inputs.{field}", values))
for name, payload in batch.bar_inputs.items():
    for field, values in payload.items():
        if isinstance(values, np.ndarray):
            rows.append(array_summary(f"bar_inputs.{name}.{field}", values))
        else:
            rows.append({"name": f"bar_inputs.{name}.{field}", "value": values})

pd.DataFrame(rows)

## Identity And Source Part Coverage

In [ ]:
def utc_from_us(ts_us: int) -> str:
    return dt.datetime.fromtimestamp(int(ts_us) / 1_000_000, tz=dt.timezone.utc).isoformat()

identity = pd.DataFrame({
    "row": np.arange(batch.sample_count),
    "ticker": batch.ticker.astype(str),
    "origin_ordinal": batch.origin_ordinal.astype(np.int64),
    "origin_timestamp_us": batch.origin_timestamp_us.astype(np.int64),
    "origin_utc": [utc_from_us(x) for x in batch.origin_timestamp_us[: batch.sample_count]],
    "source_part_key": batch.source_part_key.astype(str),
})
display(identity.head(20))

part_counts = identity.groupby("source_part_key", as_index=False).size().sort_values("size", ascending=False)
display(part_counts.head(20))

dupes = identity.duplicated(["ticker", "origin_ordinal"]).sum()
print("duplicate (ticker, origin_ordinal) rows:", int(dupes))
print("unique tickers:", int(identity["ticker"].nunique()))
print("unique source parts:", int(identity["source_part_key"].nunique()))

## Event Stream Inspection

In [ ]:
EVENT_SAMPLE_ROW = 0
event_columns = list(batch.raw_event_stream_feature_names)
print("event columns", len(event_columns), event_columns)
print("raw_event_stream shape", batch.raw_event_stream.shape)

event_stream = pd.DataFrame(batch.raw_event_stream[EVENT_SAMPLE_ROW], columns=event_columns)
display(event_stream.head(8))
display(event_stream.tail(8))

if batch.raw_event_mask.size:
    print("valid event rows for sample", EVENT_SAMPLE_ROW, int(batch.raw_event_mask[EVENT_SAMPLE_ROW].sum()), "/", batch.raw_event_mask.shape[1])

## Intraday Labels

In [ ]:
label_rows = []
for name, values in batch.intraday_labels.items():
    arr = np.asarray(values)
    label_rows.append({
        "field": name,
        "shape": tuple(arr.shape),
        "dtype": str(arr.dtype),
        "zero_fraction": float(np.mean(arr == 0)) if arr.size else None,
        "finite_fraction": float(np.mean(np.isfinite(arr))) if arr.size and np.issubdtype(arr.dtype, np.floating) else None,
    })
display(pd.DataFrame(label_rows))

LABEL_SAMPLE_ROW = 0
label_sample = {name: values[LABEL_SAMPLE_ROW] for name, values in batch.intraday_labels.items()}
display(pd.DataFrame(label_sample))
if batch.future_intraday_bar_horizons:
    print("future_intraday_bar_horizons", batch.future_intraday_bar_horizons)
    print("future_intraday_bars shape", batch.future_intraday_bars.shape)
    print("future_intraday_bar_mask true fraction", float(batch.future_intraday_bar_mask.mean()))

## Text Token Context

In [ ]:
text_rows = []
for name, payload in batch.text_inputs.items():
    item_mask = payload.get("item_mask")
    chunk_mask = payload.get("chunk_mask")
    attention_mask = payload.get("attention_mask")
    text_rows.append({
        "context": name,
        "input_ids_shape": tuple(payload["input_ids"].shape),
        "item_mask_shape": tuple(item_mask.shape) if item_mask is not None else None,
        "items_per_sample_mean": float(item_mask.sum(axis=1).mean()) if item_mask is not None and item_mask.size else 0.0,
        "chunk_true_fraction": float(chunk_mask.mean()) if chunk_mask is not None and chunk_mask.size else 0.0,
        "token_true_fraction": float(attention_mask.mean()) if attention_mask is not None and attention_mask.size else 0.0,
    })
display(pd.DataFrame(text_rows))

TEXT_SAMPLE_ROW = 0
for name, payload in batch.text_inputs.items():
    print("\n", name)
    print("item timestamps", payload.get("item_timestamp_us", np.asarray([]))[TEXT_SAMPLE_ROW])
    print("item mask", payload.get("item_mask", np.asarray([]))[TEXT_SAMPLE_ROW])
    print("chunk mask", payload.get("chunk_mask", np.asarray([]))[TEXT_SAMPLE_ROW])

## XBRL And Daily Bars

In [ ]:
if batch.xbrl_inputs:
    mask = batch.xbrl_inputs.get("mask")
    print("xbrl fields", sorted(batch.xbrl_inputs))
    print("xbrl mask shape", None if mask is None else mask.shape)
    if mask is not None and mask.size:
        print("xbrl rows per sample mean/min/max", float(mask.sum(axis=1).mean()), int(mask.sum(axis=1).min()), int(mask.sum(axis=1).max()))
        print("xbrl zero-padded samples", int(np.count_nonzero(mask.sum(axis=1) == 0)), "/", mask.shape[0])
    display(pd.DataFrame({field: values[0, :10] for field, values in batch.xbrl_inputs.items() if np.asarray(values).ndim == 2}).head(10))
else:
    print("No xbrl_inputs emitted")

bar_rows = []
for name, payload in batch.bar_inputs.items():
    values = payload.get("values")
    mask = payload.get("mask")
    bar_rows.append({
        "context": name,
        "values_shape": tuple(values.shape) if isinstance(values, np.ndarray) else None,
        "mask_shape": tuple(mask.shape) if isinstance(mask, np.ndarray) else None,
        "mask_true_fraction": float(mask.mean()) if isinstance(mask, np.ndarray) and mask.size else None,
        "feature_names": list(payload.get("feature_names", [])),
        "offsets": list(payload.get("offsets", [])),
    })
display(pd.DataFrame(bar_rows))

## Availability And State Checkpoint

In [ ]:
availability_rows = []
for name, values in batch.input_availability.items():
    arr = np.asarray(values)
    availability_rows.append({"name": name, "shape": tuple(arr.shape), "true_fraction": float(arr.mean()) if arr.size else 0.0})
display(pd.DataFrame(availability_rows))

state = loader.state_dict()
print(json.dumps(state, indent=2, sort_keys=True))